In [1]:
from dall_e import map_pixels, unmap_pixels, load_model
from dall_e import Encoder, Decoder

import torch
import torch.nn as nn
import torch.nn.functional as F

/usr/lib/python3/dist-packages/requests/__init__.py:87: RequestsDependencyWarning: urllib3 (2.7.0) or chardet (4.0.0) doesn't match a supported version!
  warnings.warn("urllib3 ({}) or chardet ({}) doesn't match a supported "


In [2]:
device = torch.device("cuda")
enc: Encoder = load_model("./agentic/checkpoints/encoder.pkl", device)
for p in enc.parameters():
    p.requires_grad = False
enc.eval()
print()

In [3]:
enc.blocks.output.conv.use_float16 = True
for param in enc.parameters():
    param.requires_grad = False
enc.eval()

# dummy_img = torch.randn(1, 3, 480, 640, device=device)
# dummy_img = torch.randn(1, 3, 120, 160, device="cuda")
dummy_img = torch.randn(1, 3, 240, 320, device=device)
torch.onnx.export(
    enc,
    dummy_img,
    "./enc.onnx",
    export_params=True,
    input_names=["pixels"],
    # output_names=[""],
    dynamic_axes=None
)
print("ONNX model saved: enc.onnx  (output: logits; argmax on host)")

/home/unitree/.local/lib/python3.10/site-packages/dall_e/encoder.py:88: TracerWarning: Converting a tensor to a Python boolean might cause the trace to be incorrect. We can't record the data flow of Python values, so this value will be treated as a constant in the future. This means that the trace might not generalize to other inputs!
  if x.shape[1] != self.input_channels:


ONNX model saved: enc.onnx  (output: logits; argmax on host)


In [ ]:
enc(dummy_img).shape

In [ ]:
# Build FP16 engine (e.g. for sanity):
# trtexec --onnx=enc.onnx --saveEngine=enc.trt --workspace=4096 --fp16
#
# Build INT8 engine (uses real OpenImages calibration; runs on Jetson Orin Nano):
# python build_int8_encoder.py

In [ ]:
# trtexec --onnx=enc.onnx --saveEngine=enc.trt --fp16 --useDLACore=0 --allowGPUFallback

In [ ]:
[05/12/2026-14:51:56] [I] === Performance summary ===
[05/12/2026-14:51:56] [I] Throughput: 13.0337 qps
[05/12/2026-14:51:56] [I] Latency: min = 58.2183 ms, max = 80.6731 ms, mean = 77.7069 ms, median = 79.987 ms, percentile(90%) = 80.2231 ms, percentile(95%) = 80.2753 ms, percentile(99%) = 80.6731 ms
[05/12/2026-14:51:56] [I] Enqueue Time: min = 0.504089 ms, max = 0.823669 ms, mean = 0.69987 ms, median = 0.71637 ms, percentile(90%) = 0.772644 ms, percentile(95%) = 0.784241 ms, percentile(99%) = 0.823669 ms
[05/12/2026-14:51:56] [I] H2D Latency: min = 0.0656738 ms, max = 0.0734863 ms, mean = 0.07128 ms, median = 0.0715942 ms, percentile(90%) = 0.072937 ms, percentile(95%) = 0.0731812 ms, percentile(99%) = 0.0734863 ms
[05/12/2026-14:51:56] [I] GPU Compute Time: min = 55.8317 ms, max = 78.3517 ms, mean = 75.3794 ms, median = 77.6631 ms, percentile(90%) = 77.9016 ms, percentile(95%) = 77.9551 ms, percentile(99%) = 78.3517 ms
[05/12/2026-14:51:56] [I] D2H Latency: min = 2.22949 ms, max = 2.32642 ms, mean = 2.25625 ms, median = 2.24997 ms, percentile(90%) = 2.25488 ms, percentile(95%) = 2.32089 ms, percentile(99%) = 2.32642 ms
[05/12/2026-14:51:56] [I] Total Host Walltime: 3.22241 s
[05/12/2026-14:51:56] [I] Total GPU Compute Time: 3.16593 s

In [ ]:
device = torch.device("cuda")
base: Decoder = load_model("./agentic/checkpoints/decoder.pkl", device)
for p in base.parameters():
    p.requires_grad = False
base.eval()
print()

In [ ]:

# dummy_img = torch.randn(1, 3, 480, 640, device=device)
# dummy_img = torch.randn(1, 3, 120, 160, device="cuda")
dummy_codewords = torch.randn(1, 8192, 30, 40, device=device)
torch.onnx.export(
    base,
    dummy_codewords,
    "./dec.onnx",
    export_params=True,
    # input_names=["pixels"],
    # output_names=[""],
    dynamic_axes=None
)
print("ONNX model saved: dec.onnx  (output: logits; argmax on host)")

In [ ]:
# trtexec --onnx=dec.onnx --saveEngine=dec.trt --fp16 --int8